# R07 — Clusters y aumento de datos con VitalDB
Este cuaderno recorre paso a paso el mismo proceso de `scripts/clusters_vitaldb.py`. Ejecútelo de arriba abajo (necesita `data/vitaldb/clinical_data.csv`; el script lo descarga).

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve() / 'scripts'))
import clusters_vitaldb as cv
import pandas as pd
ruta = Path('..') / 'data' / 'vitaldb' / 'clinical_data.csv'
df = cv.cargar(ruta)
df.shape

## 1. Limpieza
Solo adultos, rangos clínicos plausibles. No se imputa todavía (para no filtrar información de la partición de prueba).

In [ ]:
d, bit = cv.preparar(df)
bit

## 2. Separación entrenamiento / prueba por paciente
Se hace **antes** de agrupar y de generar sintéticos.

In [ ]:
split = cv.dividir_por_paciente(d, 0.2, 42)
split.value_counts()

## 3. Proceso completo: elección de k, grupos, sintéticos y calidad
El codo y la silueta justifican el k elegido (ver la tabla y las figuras).

In [ ]:
r = cv.ejecutar(df, semilla=42)
print('k =', r['k'], '| codo sugiere', r['k_codo'], '| estabilidad (ARI) =', round(r['estabilidad'], 2))
print(r['justificacion_k'])
r['tabla_k']

In [ ]:
from pathlib import Path
cv.figuras(Path('img_tmp'), r['tabla_k'], r['k'], r['k_codo'], r['datos'])
from IPython.display import Image, display
display(Image('img_tmp/clusters_k.png')); display(Image('img_tmp/clusters_perfil.png'))

## 4. Interpretación clínica de cada grupo
Los desenlaces (UCI, mortalidad) **no** se usaron para agrupar; solo describen.

In [ ]:
r['perfil']

## 5. Calidad de los sintéticos (por grupo, contra el entrenamiento real)

In [ ]:
r['calidad']

## 6. Conteo final
Todo sintético lleva `origen = sintetico` y su `cluster`; no tiene desenlaces inventados.

In [ ]:
r['final'].groupby(['cluster','origen']).size().unstack(fill_value=0)